# TradeSense · Component 03 · Layer 2 Targeted Improvements → `processed_layer2_targeted_improvements_v1`

**Scope:** two targeted questions only. Not a general audit; Phase 0 / 0b / model-ready v1 are **not** repeated. **No model is trained.**
**Frozen/never modified:** `processed_layer2_final_v3/`, `processed_layer2_model_ready_v1/` and the raw sources (SHA-256/size/mtime pinned before and re-checked after). **Output:** a new versioned folder, staged then published by atomic rename only if every assertion passes; refuses to run if it already exists.

## TASK 1 — Mt.Gox: can `buyer_7d_trade_count`, `buyer_7d_btc_sum`, `buyer_all_btc_sum` safely improve the 7-feature baseline?
For each column, evidence is gathered on **availability at prediction time, target leakage, future information, conflict reduction and reliability**. Decisions use **train and validation only** (test labels are never used). A column enters the experimental configuration only if it is **safe** *and* adds **replicated, non-fragmentation information**:

| Gate | Rule (fixed before looking at results) |
|---|---|
| Leakage | not near-deterministic (|AUC−0.5| < 0.49 on train and validation); train-fitted threshold stump has validation balanced accuracy < 0.95; within-conflict-cluster |AUC−0.5| < 0.45; no documentation line marks it forward-looking |
| Reliability | missing ≤ 1%; negatives ≤ 0.1%; internal-consistency violations ≤ 0.1% (7d count ≥ 24h count; all-time sum ≥ 7d sum; zero-count vs zero-sum) |
| Incremental information | **(I1)** separates labels *within* baseline conflict clusters, with the same sign and |AUC−0.5| ≥ 0.05 in **both** train and validation clusters, **or (I2)** resolves ≥ 5% more of the baseline conflicting rows than **row-permuted null columns** (controls for the trivial fragmentation effect of adding features) |
| Availability | cannot be proven without a definition: reported as **UNVERIFIED** unless documentation exists; an admitted column is therefore **experimental only** |

The baseline 7-feature configuration and the frozen parquets are **never overwritten**; the experimental config adds a separate small parquet of the admitted columns, scaled with a **train-only** fit. The **EXPLORATORY verdict is preserved** unless chronology *and* label provenance are recovered from the source/documentation (they are searched for here, never invented).

## TASK 2 — Sapienza: what exactly are the category-B "potentially harmful" duplicate rows?
Every B row is decomposed by the *reason* it was flagged (non-exclusive and exclusive attribution), then re-typed as: **actual duplicate source record**, **repeated legitimate market window (same market)**, **repeated vector across markets (coincidence)** or **cross-split contamination candidate**. Rows are never deleted merely because vectors are identical. An **optional training-only configuration** is created only if a pre-registered rule is met (giant, degenerate, label-pure, protected-row-free clusters capped at K=20 copies, ≥ 1% of train); validation/test and the primary benchmark stay untouched.

## TASK 3 — Targeted validation of the NEW artifacts only
Assertions on alignment with v1, row counts, attribution sums, filter safety, scaler train-only fit, leakage-gate reproducibility, frozen/v1 immutability and manifest readability. If an issue cannot be resolved from the available evidence it is **documented and investigation stops**.

## 0 · Setup, pre-registered constants, pinning

In [ ]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
import sklearn, joblib
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda v: f'{v:,.6g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> set TRADESENSE_BASE to your local dataset root.')

BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
V3 = BASE / 'processed_layer2_final_v3'                       # FROZEN
V1 = BASE / 'processed_layer2_model_ready_v1'                 # FROZEN (created by notebook 01)
OUT_FINAL = BASE / 'processed_layer2_targeted_improvements_v1'
STAGE = BASE / ('.layer2_targeted_stage_' + uuid.uuid4().hex[:8])
SAP_FILE = BASE / 'pump-and-dump-dataset-master/labeled_features/features_5S.csv/features_5S.csv'
GOX_FILE = BASE / 'gox_ml_samples.csv'
SAP_DIR = BASE / 'pump-and-dump-dataset-master'
for p in (V3, V1): assert p.is_dir(), f'required input missing: {p}'
for p in (SAP_FILE, GOX_FILE): assert p.is_file(), f'raw source missing: {p}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
RUN_STARTED = datetime.now(timezone.utc).isoformat(); SEED = 42
print('base:', BASE, '| frozen v3 + v1 found | output (new):', OUT_FINAL.name, '| stage:', STAGE.name)

SAP_BASE = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price', 'avg_price', 'avg_price_max', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']
SAP_ACTIVITY = ['std_rush_order', 'avg_rush_order', 'std_trades', 'std_volume', 'avg_volume', 'std_price']
GOX_BASE = ['bitcoins', 'price_deviation', 'buyer_24h_trade_count', 'seller_24h_trade_count']
TARGET_COLS = ['buyer_7d_trade_count', 'buyer_7d_btc_sum', 'buyer_all_btc_sum']
SPL = ('train', 'val', 'test')

# ---------------- pre-registered decision constants (documented in the intro table) ----------------
CFG = {'seed': SEED, 'sample_rows': 400_000, 'null_perms_single': 3, 'null_perms_set': 5,
       'auc_near_deterministic_dev': 0.49, 'stump_ba_leak': 0.95, 'within_cluster_leak_dev': 0.45,
       'missing_max': 0.01, 'negative_max': 0.001, 'violation_max': 0.001,
       'replicate_dev': 0.05, 'min_mixed_rows_each_split': 200, 'excess_conflict_share': 0.05,
       'dedup_cap_K': 20, 'dedup_min_share_of_train': 0.01}

In [ ]:
def sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def hash_rows(df, cols): return pd.util.hash_pandas_object(df[cols], index=False).to_numpy()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def class_counts(y):
    y = np.asarray(y).astype(int); pos, neg = int(y.sum()), int((1 - y).sum())
    return {'rows': pos + neg, 'positives': pos, 'negatives': neg, 'scale_pos_weight_full': (neg / pos) if pos else None}
def scan_docs(root, max_depth=3, max_files=6, max_bytes=400_000):
    found = []
    if not root.is_dir(): return found
    d0 = len(root.parts)
    for dp, dn, fn in os.walk(root):
        if len(Path(dp).parts) - d0 >= max_depth: dn[:] = []
        for f in sorted(fn):
            if re.search(r'readme|licen[cs]e|citation|\.md$|\.rst$|\.txt$', f, re.I):
                p = Path(dp) / f
                try:
                    if p.stat().st_size <= max_bytes: found.append(p)
                except OSError: pass
        if len(found) >= max_files: break
    return found[:max_files]

# ---------------- pin frozen inputs ----------------
t0 = time.time()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
PIN_V3, PIN_V1 = pin_dir(V3), pin_dir(V1)
PIN_SRC = {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in (SAP_FILE, GOX_FILE)}
print(f'pinned v3 ({len(PIN_V3)} files) and model-ready v1 ({len(PIN_V1)} files) in {time.time()-t0:.0f}s')
M1 = {ds: json.loads((V1 / ds / 'training_manifest.json').read_text()) for ds in ('sapienza', 'mtgox')}
print('v1 verdicts:', {ds: M1[ds]['verdict'] for ds in M1}, '| v1 validation:', {ds: M1[ds].get('validation') for ds in M1})

## 1 · TASK 1 — Mt.Gox: the three excluded source columns
Decisions use **rows from the train and validation partitions only** (row index < end of validation); the test partition never enters any gate. Test labels are used nowhere in this section.

In [ ]:
GOX_M = joblib.load(V3 / 'mtgox_metadata.joblib'); GOX_FEATS = list(GOX_M['features']); GOX_T = 'wash'
GOX_PQ = {sp: pd.read_parquet(V3 / f'mtgox_{sp}.parquet') for sp in SPL}
assert list(GOX_PQ['train'].columns) == GOX_FEATS + [GOX_T]
GOX_N = {sp: len(v) for sp, v in GOX_PQ.items()}; N_GOX = sum(GOX_N.values()); A_IDX = GOX_N['train']; B_IDX = A_IDX + GOX_N['val']
BOUNDS = {'train': (0, A_IDX), 'val': (A_IDX, B_IDX), 'test': (B_IDX, N_GOX)}
HDR = list(pd.read_csv(GOX_FILE, nrows=0).columns)
PRESENT = [c for c in TARGET_COLS if c in HDR]; ABSENT = [c for c in TARGET_COLS if c not in HDR]
print('source columns:', HDR); print('requested columns present:', PRESENT, '| absent:', ABSENT)

TIME_RE = re.compile(r'time|date|stamp|^ts$|epoch|^year$|^month$|^day$|^hour$', re.I)
PART_RE = re.compile(r'(^|_)(user|trader|account|wallet|address|participant|counterpart\w*|buyer|seller|party)(_id)?$', re.I)
GEN_ID_RE = re.compile(r'(^|_)(id|uid|uuid|guid|hash)$', re.I)
EXTRA_TIME = [c for c in HDR if TIME_RE.search(c) and c not in GOX_BASE + [GOX_T] + TARGET_COLS]
GEN_IDS = [c for c in HDR if GEN_ID_RE.search(c) and c not in GOX_BASE + [GOX_T] + TARGET_COLS]
PARTICIPANT = [c for c in HDR if PART_RE.search(c) and c not in GOX_BASE + [GOX_T] + TARGET_COLS]
usecols = list(dict.fromkeys(GOX_BASE + [GOX_T] + PRESENT + EXTRA_TIME + GEN_IDS + PARTICIPANT))
probe = pd.read_csv(GOX_FILE, nrows=50_000, usecols=usecols)
dmap = {c: ('float64' if (c in GOX_BASE + [GOX_T] + EXTRA_TIME + GEN_IDS + PARTICIPANT) and pd.api.types.is_numeric_dtype(probe[c]) else ('float32' if pd.api.types.is_numeric_dtype(probe[c]) else 'category')) for c in usecols}
GX = pd.read_csv(GOX_FILE, usecols=usecols, dtype=dmap); del probe
assert len(GX) == N_GOX, f'source rows {len(GX)} != frozen total {N_GOX}'
print(f'loaded {len(GX):,} rows x {len(usecols)} columns ({GX.memory_usage(deep=True).sum()/1e6:.0f} MB)')

def gox_features(d):                                   # copied from the frozen Layer 2 pipeline
    x = d[GOX_BASE].apply(pd.to_numeric, errors='coerce').copy()
    x['trade_count_ratio'] = x['buyer_24h_trade_count'] / (x['seller_24h_trade_count'] + 1)
    x['log_bitcoins'] = np.log1p(x['bitcoins'].clip(lower=0)); x['abs_price_deviation'] = x['price_deviation'].abs()
    return x.replace([np.inf, -np.inf], np.nan)

# ---- alignment of the raw source with the frozen split (row i of source == row i of concatenated v3 parquets) ----
Hs, ALIGN_OK = [], {}
for sp, (lo, hi) in BOUNDS.items():
    part = GX.iloc[lo:hi]; arr = GOX_M['scaler'].transform(GOX_M['imputer'].transform(gox_features(part))).astype(np.float32); saved = GOX_PQ[sp][GOX_FEATS].to_numpy(np.float32)
    ALIGN_OK[sp] = {'features_bit_identical': bool(np.array_equal(arr, saved)), 'labels_identical': bool(np.array_equal(part[GOX_T].to_numpy().astype(np.int8), GOX_PQ[sp][GOX_T].to_numpy().astype(np.int8)))}
    assert all(ALIGN_OK[sp].values()), f'source/frozen misalignment in {sp}: {ALIGN_OK[sp]}'
    Hs.append(hash_rows(GOX_PQ[sp], GOX_FEATS)); del arr, saved
H = np.concatenate(Hs); del Hs
Y = np.concatenate([GOX_PQ[sp][GOX_T].to_numpy().astype(np.int8) for sp in SPL])
BASE_COUNTS = {sp: class_counts(GOX_PQ[sp][GOX_T]) for sp in SPL}
del GOX_PQ
print('source rows align bit-for-bit with frozen v3 (features and labels):', ALIGN_OK)

In [ ]:
# ---------------- documentation / timestamps / participant identifiers (searched, never invented) ----------------
GOX_DOC_DIRS = [p for p in BASE.iterdir() if p.is_dir() and re.search(r'gox', p.name, re.I)]
DOC_FILES = [p for d in GOX_DOC_DIRS for p in scan_docs(d)] + [p for p in BASE.iterdir() if p.is_file() and re.search(r'gox', p.name, re.I) and re.search(r'readme|\.md$|\.txt$|\.rst$|licen', p.name, re.I)]
KEY = re.compile(r'wash|label|7d|all_btc|btc_sum|trade_count|window|trailing|rolling|past|future|forward|look|time ?stamp|participant|user|trader|ground|heuristic|derived|defin', re.I)
ALL_LINES, DOC_HITS = {}, {}
for p in DOC_FILES:
    ls = p.read_text(encoding='utf-8', errors='replace').splitlines(); rel = str(p.relative_to(BASE)); ALL_LINES[rel] = ls
    DOC_HITS[rel] = [l.strip()[:240] for l in ls if KEY.search(l)][:40]
    print(f'\n--- {rel} ({len(ls)} lines) ---'); [print('  ', h) for h in DOC_HITS[rel]]
print('documentation files near the Mt.Gox source:', [str(p.relative_to(BASE)) for p in DOC_FILES] or 'NONE')

def assess_time_col(s, n_probe=100_000):
    n = len(s); idx = np.linspace(0, n - 1, min(n, n_probe)).astype(int); v = s.iloc[idx]
    if str(s.dtype) in ('category', 'object', 'string'):
        t = pd.to_datetime(v.astype(str), errors='coerce', utc=True); arr = t.dropna().astype('int64').to_numpy(); ok = float(t.notna().mean())
    else:
        x = pd.to_numeric(v, errors='coerce'); ok = float(x.notna().mean()); arr = x.dropna().to_numpy(float)
    mono = float((np.diff(arr) >= 0).mean()) if len(arr) > 1 else float('nan'); nun = int(pd.Series(arr).nunique())
    return {'parse_ok_share': ok, 'monotonic_non_decreasing_share_sampled': mono, 'n_unique_in_sample': nun, 'reliable_chronology': bool(ok >= 0.99 and mono >= 0.999 and nun >= 1000)}
TIME_EVID = {c: assess_time_col(GX[c]) for c in EXTRA_TIME}
CHRONO_SUPPORTED = any(v['reliable_chronology'] for v in TIME_EVID.values())
LABEL_DOC_LINES = [l for ls in ALL_LINES.values() for l in ls if re.search(r'wash', l, re.I) and re.search(r'defin|label|flag|deriv|heurist|detect|self|round|ground', l, re.I)]
LABEL_DOC_FOUND = bool(LABEL_DOC_LINES)
PRIOR = {}
pp = V1 / 'mtgox' / 'reports' / 'label_provenance_and_classification_report.json'
if pp.is_file(): PRIOR = json.loads(pp.read_text())
print('\ntime-like columns:', EXTRA_TIME or 'NONE', json.dumps(TIME_EVID)); print('participant-like columns:', PARTICIPANT or 'NONE', '| generic id columns (not participant evidence):', GEN_IDS or 'NONE')
print('label-definition documentation lines mentioning wash:', LABEL_DOC_LINES[:6] or 'NONE'); print('v1 classification (for reference):', PRIOR.get('classification'))
PROVENANCE = {'documentation_files': [str(p.relative_to(BASE)) for p in DOC_FILES], 'time_columns': TIME_EVID, 'chronology_supported': CHRONO_SUPPORTED, 'participant_like_columns': PARTICIPANT, 'generic_id_columns': GEN_IDS,
              'label_definition_lines_found': LABEL_DOC_LINES[:10], 'label_documented': LABEL_DOC_FOUND, 'v1_classification': PRIOR.get('classification')}

In [ ]:
# ---------------- per-column assessment: availability, leakage, future information, reliability ----------------
rng = np.random.default_rng(SEED)
TR_IDX = np.sort(rng.choice(A_IDX, size=min(A_IDX, CFG['sample_rows']), replace=False)); VA_IDX = A_IDX + np.sort(rng.choice(B_IDX - A_IDX, size=min(B_IDX - A_IDX, CFG['sample_rows']), replace=False))
DECISION_MAX_ROW = int(max(TR_IDX.max(), VA_IDX.max())); assert DECISION_MAX_ROW < B_IDX
FWD_RE = re.compile(r'future|forward|next|following|after|subsequent|look-?ahead', re.I); TRAIL_RE = re.compile(r'trailing|past|previous|prior|before|preceding|rolling|until|up to|last \d+', re.I)

def stump_ba(xtr, ytr, xva, yva):
    qs = np.unique(np.quantile(xtr[np.isfinite(xtr)], np.linspace(.005, .995, 199))); best = (-1.0, None, 1)
    def ba(x, y, t, d):
        pred = (d * (x - t) > 0); pos = max(int((y == 1).sum()), 1); neg = max(int((y == 0).sum()), 1)
        return (((pred & (y == 1)).sum() / pos) + (((~pred) & (y == 0)).sum() / neg)) / 2
    for t in qs:
        for d in (1, -1):
            b = ba(xtr, ytr, t, d)
            if b > best[0]: best = (b, t, d)
    return {'train_balanced_acc': float(best[0]), 'threshold': None if best[1] is None else float(best[1]), 'direction': best[2], 'val_balanced_acc': float(ba(xva, yva, best[1], best[2])) if best[1] is not None else float('nan')}

def relations(c):
    x = GX[c].to_numpy(float); out = {}
    if c == 'buyer_7d_trade_count':
        out['7d_count < 24h_count (nested windows violated)'] = float((x < GX['buyer_24h_trade_count'].to_numpy(float)).mean())
        if 'buyer_7d_btc_sum' in GX: out['count==0 but 7d_btc_sum>0'] = float(((x == 0) & (GX['buyer_7d_btc_sum'].to_numpy(float) > 0)).mean())
    if c == 'buyer_7d_btc_sum':
        if 'buyer_7d_trade_count' in GX: out['sum>0 but 7d_count==0'] = float(((x > 0) & (GX['buyer_7d_trade_count'].to_numpy(float) == 0)).mean())
    if c == 'buyer_all_btc_sum' and 'buyer_7d_btc_sum' in GX:
        s7 = GX['buyer_7d_btc_sum'].to_numpy(float); out['all_sum < 7d_sum (nested windows violated)'] = float((x < s7 - (1e-4 * np.abs(s7) + 1e-6)).mean())
    return out

COL = {}
for c in PRESENT:
    x = GX[c].to_numpy(float); xt, yt, xv, yv = x[TR_IDX], Y[TR_IDX], x[VA_IDX], Y[VA_IDX]
    a_tr, a_va = auc_rank(xt, yt), auc_rank(xv, yv); st = stump_ba(xt, yt, xv, yv); rel = relations(c)
    blocks = [np.nanmedian(b) for b in np.array_split(x, 10)]
    doc_lines = [l.strip()[:240] for ls in ALL_LINES.values() for l in ls if c.lower() in l.lower()]
    avail = ('DOCUMENTED_FORWARD_LOOKING' if any(FWD_RE.search(l) for l in doc_lines) else 'DOCUMENTED_TRAILING (human confirmation required)' if any(TRAIL_RE.search(l) for l in doc_lines)
             else ('UNVERIFIED (documentation mentions the column without a temporal definition)' if doc_lines else 'UNVERIFIED (no documentation of this column)'))
    COL[c] = {'reliability': {'missing_share': float(np.isnan(x).mean()), 'negative_share': float(np.nanmean(x < 0)), 'inf_share': float(np.isinf(x).mean()), 'n_unique': int(pd.Series(x).nunique()), 'zero_share': float(np.nanmean(x == 0)),
                              'integer_valued_share': float(np.nanmean(np.isclose(x, np.round(x)))), 'relation_violations': rel, 'max_relation_violation': max([x for k, x in rel.items() if 'nested' in k] or [0.0]), 'note_relations': 'only nested-window relations gate; zero-count/zero-sum cross-checks are informational (a window sum may include the current trade)'},
              'leakage': {'auc_train_sample': a_tr, 'auc_val_sample': a_va, 'dev_train': abs(a_tr - .5) if a_tr == a_tr else float('nan'), 'dev_val': abs(a_va - .5) if a_va == a_va else float('nan'), 'stump': st},
              'availability_at_prediction_time': avail, 'documentation_lines': doc_lines[:8],
              'row_order_trend': {'block_medians': [float(b) for b in blocks], 'spearman_vs_block': float(stats.spearmanr(np.arange(10), blocks)[0]), 'note': 'monotone growth with row order would suggest a cumulative/all-time aggregate, but chronology of the file is unverified'}}
rows = [{'column': c, 'missing': v['reliability']['missing_share'], 'negative': v['reliability']['negative_share'], 'max_relation_viol': v['reliability']['max_relation_violation'], 'AUC_train': v['leakage']['auc_train_sample'], 'AUC_val': v['leakage']['auc_val_sample'],
         'stump_BA_val': v['leakage']['stump']['val_balanced_acc'], 'trend_rho': v['row_order_trend']['spearman_vs_block'], 'availability': v['availability_at_prediction_time'][:45]} for c, v in COL.items()]
display(pd.DataFrame(rows).set_index('column') if rows else 'no requested column present')
for c, v in COL.items(): print(c, 'relations:', v['reliability']['relation_violations'])

In [ ]:
# ---------------- conflict analysis: baseline 7-feature clusters vs expanded, with row-permuted null controls ----------------
def mixed_mask(h, y):
    d = pd.DataFrame({'h': h, 'y': np.asarray(y).astype(np.int64)}); g = d.groupby('h')['y']
    n = g.transform('size').to_numpy(); s = g.transform('sum').to_numpy()
    return (n > 1) & (s > 0) & (s < n), n

mix_dec, _n = mixed_mask(H[:B_IDX], Y[:B_IDX]); M_DEC = np.flatnonzero(mix_dec)
mix_all, _n = mixed_mask(H, Y); M_ALL = np.flatnonzero(mix_all); del _n
def n_clusters(M): return int(pd.Series(H[M]).nunique()) if len(M) else 0
BASE_CONFLICT = {'decision_rows(train+val)': {'conflicting_rows': int(len(M_DEC)), 'conflicting_clusters': n_clusters(M_DEC), 'share_of_rows': float(len(M_DEC) / B_IDX)},
                 'all_rows(descriptive)': {'conflicting_rows': int(len(M_ALL)), 'conflicting_clusters': n_clusters(M_ALL), 'share_of_rows': float(len(M_ALL) / N_GOX)}}
print('BASELINE 7-feature conflicts:', json.dumps(BASE_CONFLICT))

def expand_stats(M, cols, perms, seed):
    if len(M) == 0: return {'rows': 0}
    base_vals = {c: GX[c].to_numpy()[M] for c in cols}
    def one(vals):
        hh = pd.util.hash_pandas_object(pd.DataFrame({'h': H[M], **vals}), index=False).to_numpy(); mm, nn = mixed_mask(hh, Y[M])
        return int(mm.sum()), int(((~mm) & (nn >= 2)).sum()), int(((~mm) & (nn == 1)).sum()), int(pd.Series(hh[mm]).nunique()) if mm.any() else 0
    o = one(base_vals); r = np.random.default_rng(seed); nl = [one({c: r.permutation(v) for c, v in base_vals.items()}) for _ in range(perms)]
    nm = np.array(nl, float).mean(axis=0); L = len(M)
    return {'baseline_conflicting_rows': int(L), 'still_conflicting_rows': o[0], 'still_conflicting_clusters': o[3], 'resolved_into_pure_nonsingleton_clusters': o[1], 'isolated_as_singletons': o[2], 'share_resolved': 1 - o[0] / L,
            'null_share_resolved_mean': float(1 - nm[0] / L), 'null_isolated_singletons_mean': float(nm[2]), 'excess_share_resolved_vs_null': float((1 - o[0] / L) - (1 - nm[0] / L)), 'excess_pure_nonsingleton_vs_null': float((o[1] - nm[1]) / L)}

# within-conflict-cluster association (cluster mean of the column removed; labels never used to demean)
WIN = {}
if len(M_DEC):
    hm = H[M_DEC]; tr_m = M_DEC < A_IDX; va_m = ~tr_m
    for c in PRESENT:
        v = GX[c].to_numpy(float)[M_DEC]; vc = v - pd.Series(v).groupby(hm).transform('mean').to_numpy(); yy = Y[M_DEC]
        a1, a2 = auc_rank(vc[tr_m], yy[tr_m]), auc_rank(vc[va_m], yy[va_m])
        WIN[c] = {'auc_train_clusters': a1, 'auc_val_clusters': a2, 'n_train': int(tr_m.sum()), 'n_val': int(va_m.sum()),
                  'dev_train': abs(a1 - .5) if a1 == a1 else float('nan'), 'dev_val': abs(a2 - .5) if a2 == a2 else float('nan'), 'same_sign': bool(a1 == a1 and a2 == a2 and (a1 - .5) * (a2 - .5) > 0)}
SINGLE = {c: expand_stats(M_DEC, [c], CFG['null_perms_single'], SEED + i) for i, c in enumerate(PRESENT)}
display(pd.DataFrame(WIN).T if WIN else 'no conflicts in train+val')
display(pd.DataFrame(SINGLE).T if SINGLE else '')

In [ ]:
# ---------------- admission gates (pre-registered; train+val only) ----------------
GATES = {}
for c in PRESENT:
    v, w, s = COL[c], WIN.get(c), SINGLE[c]
    leak = {'no_forward_doc': 'FORWARD' not in v['availability_at_prediction_time'], 'auc_not_near_deterministic': bool(v['leakage']['dev_train'] < CFG['auc_near_deterministic_dev'] and v['leakage']['dev_val'] < CFG['auc_near_deterministic_dev']),
            'stump_not_predictive': bool(v['leakage']['stump']['val_balanced_acc'] < CFG['stump_ba_leak']),
            'within_cluster_not_deterministic': bool(w is None or (w['dev_train'] != w['dev_train']) or (w['dev_train'] < CFG['within_cluster_leak_dev'] and w['dev_val'] < CFG['within_cluster_leak_dev']))}
    rel = {'missing_ok': v['reliability']['missing_share'] <= CFG['missing_max'], 'negative_ok': v['reliability']['negative_share'] <= CFG['negative_max'], 'consistency_ok': v['reliability']['max_relation_violation'] <= CFG['violation_max']}
    I1 = bool(w and w['same_sign'] and w['dev_train'] >= CFG['replicate_dev'] and w['dev_val'] >= CFG['replicate_dev'] and w['n_train'] >= CFG['min_mixed_rows_each_split'] and w['n_val'] >= CFG['min_mixed_rows_each_split'])
    I2 = bool(s.get('excess_share_resolved_vs_null', -1) >= CFG['excess_conflict_share'])
    safe = all(leak.values()) and all(rel.values())
    GATES[c] = {'leakage_gates': leak, 'reliability_gates': rel, 'I1_replicated_within_cluster_signal': I1, 'I2_excess_conflict_resolution_vs_null': I2, 'safe': bool(safe), 'informative': bool(I1 or I2),
                'admitted_to_experimental': bool(safe and (I1 or I2)), 'availability_at_prediction_time': v['availability_at_prediction_time']}
    why = [k for k, ok in {**leak, **rel}.items() if not ok] + ([] if (I1 or I2) else ['no incremental information (I1/I2 both failed)'])
    GATES[c]['rejection_reasons'] = why
    print(('ADMITTED ' if GATES[c]['admitted_to_experimental'] else 'REJECTED '), c, '|', why or 'all gates passed', '| availability:', v['availability_at_prediction_time'])
ADMITTED = [c for c in PRESENT if GATES[c]['admitted_to_experimental']]
ALL3_DEC = expand_stats(M_DEC, PRESENT, CFG['null_perms_set'], SEED + 11) if PRESENT else {}
ADM_DEC = expand_stats(M_DEC, ADMITTED, CFG['null_perms_set'], SEED + 12) if ADMITTED else {}
ADM_ALL = expand_stats(M_ALL, ADMITTED, CFG['null_perms_set'], SEED + 13) if ADMITTED else {}
ALL3_ALL = expand_stats(M_ALL, PRESENT, CFG['null_perms_set'], SEED + 14) if PRESENT else {}
CONFLICT_TABLE = {'baseline_7': BASE_CONFLICT, 'plus_each_single_column(train+val rows)': SINGLE, 'plus_all_requested_columns(train+val rows)': ALL3_DEC, 'plus_all_requested_columns(all rows, descriptive)': ALL3_ALL,
                  'plus_admitted_columns(train+val rows)': ADM_DEC, 'plus_admitted_columns(all rows, descriptive)': ADM_ALL}
print('\nADMITTED columns:', ADMITTED or 'NONE'); print(json.dumps({k: v for k, v in CONFLICT_TABLE.items() if k != 'plus_each_single_column(train+val rows)'}, indent=1, default=jsonable)[:3500])

In [ ]:
# ---------------- experimental feature configuration (only if columns were admitted); baseline untouched ----------------
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
EXP = None
if ADMITTED:
    Xe = GX[ADMITTED].to_numpy(np.float64)
    imp_e = SimpleImputer(strategy='median', keep_empty_features=True).fit(Xe[:A_IDX]); sc_e = RobustScaler().fit(imp_e.transform(Xe[:A_IDX]))      # TRAIN rows only
    EXP = {'columns': ADMITTED, 'imputer': imp_e, 'scaler': sc_e, 'arrays': {sp: sc_e.transform(imp_e.transform(Xe[lo:hi])).astype(np.float32) for sp, (lo, hi) in BOUNDS.items()}}
    assert all(np.isfinite(a).all() for a in EXP['arrays'].values()); del Xe
    print('experimental extra columns:', ADMITTED, '| scaler fitted on', A_IDX, 'train rows only | center', sc_e.center_, 'scale', sc_e.scale_)
else:
    print('No column was admitted -> NO experimental configuration is created; the 7-feature baseline stays the only Mt.Gox configuration.')
GOX_VERDICT = 'EXPLORATORY' + (' (upgrade candidate: chronology AND label definition found in documentation - human confirmation required)' if (CHRONO_SUPPORTED and LABEL_DOC_FOUND) else '')
GOX_VERDICT_REASONS = ([] if CHRONO_SUPPORTED else ['no reliable time column -> chronology of the row-index split NOT recoverable']) + ([] if LABEL_DOC_FOUND else ['no documentation defining the wash label (observed vs estimated vs derived) was found']) + \
                      ([] if PARTICIPANT else ['no participant identifier exists (activity counts are not identifiers)']) + ['conflicting-label clusters remain (%d rows in train+val under the baseline; %s with the admitted columns)' % (BASE_CONFLICT['decision_rows(train+val)']['conflicting_rows'], ADM_DEC.get('still_conflicting_rows', 'n/a - none admitted'))]
print('MT.GOX VERDICT:', GOX_VERDICT); [print('  -', r) for r in GOX_VERDICT_REASONS]

## 2 · TASK 2 — Sapienza: the 253k category-B "potentially harmful" duplicate observations
Nothing is replayed from scratch: the **v1 sidecars** (row-aligned, already validated by notebook 01) are loaded, and every link in the chain is re-proven here before it is used: `row_id` = positional index, `gt` equal to the frozen parquet, `vec_hash` equal to a fresh hash of the frozen features, and the v1 duplicate category **reproduced exactly** from the raw source. If any of these fail, the notebook aborts.

In [ ]:
SAP_M = joblib.load(V3 / 'sapienza_metadata.joblib'); SAP_FEATS = list(SAP_M['features']); SAP_T = 'gt'
SAP_PQ = {sp: pd.read_parquet(V3 / f'sapienza_{sp}.parquet') for sp in SPL}
SC = {sp: pd.read_parquet(V1 / 'sapienza' / 'metadata' / f'sapienza_{sp}_sidecar.parquet') for sp in SPL}
for sp in SPL:
    s, q = SC[sp], SAP_PQ[sp]
    assert len(s) == len(q), f'{sp}: sidecar rows {len(s)} != frozen rows {len(q)}'
    assert np.array_equal(s['row_id'].to_numpy(), np.arange(len(q))), f'{sp}: row_id is not the positional index'
    assert np.array_equal(s['gt'].to_numpy().astype(np.int8), q[SAP_T].to_numpy().astype(np.int8)), f'{sp}: sidecar gt != frozen target'
    assert np.array_equal(s['vec_hash'].to_numpy(np.uint64), hash_rows(q, SAP_FEATS)), f'{sp}: sidecar vec_hash != hash of frozen features'
print('v1 sidecars re-proven aligned to frozen v3:', {sp: len(SC[sp]) for sp in SPL})

# ---- raw source: identical-source-record hash (all raw columns) and raw activity, exactly as notebook 01 ----
src = pd.read_csv(SAP_FILE)
SRC_COLS = list(src.columns)
SREC_ALL = pd.util.hash_pandas_object(src, index=False).to_numpy()
act = src[SAP_ACTIVITY].apply(pd.to_numeric, errors='coerce').fillna(0)
INACT_ALL = (act == 0).all(axis=1).to_numpy(); del act
print(f'raw source rows {len(src):,}; columns {SRC_COLS}')
del src

frames = []
for i, sp in enumerate(SPL):
    s = SC[sp]; sr = s['source_row_id'].to_numpy(np.int64)
    frames.append(pd.DataFrame({'split': sp, 'split_code': i, 'row_id': s['row_id'].to_numpy(np.int32), 'source_row_id': sr, 'h': s['vec_hash'].to_numpy(np.uint64),
                                'gt': s['gt'].to_numpy(np.int8), 'pump_index': s['pump_index'].to_numpy(float), 'n_missing': s['n_missing_features'].to_numpy(np.int8),
                                'inactive': INACT_ALL[sr], 'coin': s['coin'].astype('string').fillna('<NA>').to_numpy(), 'date': pd.to_datetime(s['date'], utc=True),
                                'srec_h': SREC_ALL[sr], 'v1_cat': s['dup_category'].astype('string').to_numpy()}))
G = pd.concat(frames, ignore_index=True); del frames
G['is_event'] = G.pump_index >= 0
g = G.groupby('h')
G['cluster_n'] = g['h'].transform('size'); G['cluster_pos'] = g['gt'].transform('sum'); G['cluster_nsplits'] = g['split_code'].transform('nunique')
G['cluster_event'] = g['is_event'].transform('max'); G['cluster_ncoins'] = g['coin'].transform('nunique'); G['cluster_all_inactive'] = g['inactive'].transform('min')
G['cluster_mixed'] = (G.cluster_pos > 0) & (G.cluster_pos < G.cluster_n)
gs = G.groupby('srec_h')
G['srec_n'] = gs['srec_h'].transform('size'); G['srec_rank'] = gs.cumcount(); G['srec_nsplits'] = gs['split_code'].transform('nunique'); G['srec_in_eval'] = gs['split_code'].transform('max') > 0
rep = G.cluster_n >= 2
G['dup_category'] = np.select([~rep, rep & (G['gt'] == 1) & (G.cluster_nsplits > 1),
                               rep & ((G['gt'] == 1) | G.is_event | G.cluster_mixed | G.cluster_event.astype(bool) | (G.srec_n >= 2)),
                               rep & (G.n_missing >= 1), rep & G.inactive, rep],
                              ['none', 'D_cross_split_positive', 'B_potentially_harmful', 'C_imputation_driven', 'A_inactive_background', 'A2_other_background_repeat'], default='none')
assert (G.dup_category.to_numpy() == G.v1_cat.to_numpy().astype(str)).all(), 'recomputed duplicate category != v1 sidecar category (raw source / v1 logic mismatch)'
CAT_TAB = G.groupby(['dup_category', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPL), fill_value=0); CAT_TAB['total'] = CAT_TAB.sum(axis=1)
print('v1 category reproduced EXACTLY from the raw source for all', len(G), 'rows'); display(CAT_TAB)
Bm = (G.dup_category == 'B_potentially_harmful'); N_B = int(Bm.sum())
print('category-B rows:', N_B, '(user-reported v1 figure: 253,407)')

### 2a · Why exactly were the B rows flagged?
The v1 B rule is an **OR** of five conditions on a repeated vector. They are reported (i) as independent flags, and (ii) as an **exclusive attribution in priority order** (the first condition that applies), so the "only reason" count shows how many rows are B *only by association*.

In [ ]:
Bd = G[Bm].copy()
Bd['f_identical_source_record'] = Bd.srec_n >= 2
Bd['f_positive_row'] = Bd['gt'] == 1
Bd['f_event_window_row'] = Bd.is_event
Bd['f_label_mixed_cluster'] = Bd.cluster_mixed
Bd['f_cluster_contains_event_rows'] = Bd.cluster_event.astype(bool)
FL = [c for c in Bd.columns if c.startswith('f_')]
PRIO = ['f_identical_source_record', 'f_positive_row', 'f_event_window_row', 'f_label_mixed_cluster', 'f_cluster_contains_event_rows']
Bd['b_primary_reason'] = np.select([Bd[c] for c in PRIO], [c[2:] for c in PRIO], default='none')
assert (Bd.b_primary_reason != 'none').all(), 'a B row without any B condition'
INDEP = pd.DataFrame({c[2:]: Bd.groupby('split')[c].sum() for c in FL}).reindex(SPL).T; INDEP['total'] = INDEP.sum(axis=1)
EXCL_ATTR = Bd.groupby(['b_primary_reason', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPL), fill_value=0); EXCL_ATTR['total'] = EXCL_ATTR.sum(axis=1)
EXCL_ATTR = EXCL_ATTR.reindex([c[2:] for c in PRIO]).fillna(0).astype(int)
assert int(EXCL_ATTR['total'].sum()) == N_B, 'exclusive attribution does not sum to the B total'
print('(i) independent flags (a row can carry several):'); display(INDEP)
print('(ii) exclusive attribution (priority order):'); display(EXCL_ATTR)
ONLY_ASSOC = int(EXCL_ATTR.loc['cluster_contains_event_rows', 'total'])
print(f'rows flagged B ONLY because their vector also occurs in an event row: {ONLY_ASSOC:,} ({ONLY_ASSOC/N_B:.1%} of B)')
PAT = Bd[FL].astype(int).astype(str).agg(''.join, axis=1).value_counts().head(10).rename('rows').to_frame(); PAT.index.name = 'pattern ' + ','.join(c[2:] for c in FL); display(PAT)

### 2b · Re-typing every B row: what is it really?
Exclusive types, assigned in this order. Only the first two are genuine data-integrity concerns; the others are repeated *observations* or *vectors*.

| Type | Definition |
|---|---|
| `actual_duplicate_source_record` | all raw CSV columns (incl. timestamp and coin) identical to another row |
| `cross_split_contamination_candidate` | vector also occurs in another split **and** the row is an event/positive/label-mixed row |
| `label_conflict_cluster` | identical vector carries both labels |
| `positive_repeat` | positive row whose vector repeats within one split |
| `event_window_repeat_same_market` / `_cross_market` | event-window row (pre-pump) with a repeated vector; same coin vs several coins |
| `background_flagged_by_association` | negative, non-event row; flagged only because the vector also appears in an event row elsewhere |

In [ ]:
sa = Bd.srec_n >= 2
xs = (Bd.cluster_nsplits > 1) & (Bd.f_event_window_row | Bd.f_positive_row | Bd.f_label_mixed_cluster)
Bd['b_type'] = np.select([sa, xs, Bd.f_label_mixed_cluster, Bd.f_positive_row, Bd.f_event_window_row & (Bd.cluster_ncoins <= 1), Bd.f_event_window_row],
                         ['actual_duplicate_source_record', 'cross_split_contamination_candidate', 'label_conflict_cluster', 'positive_repeat', 'event_window_repeat_same_market', 'event_window_repeat_cross_market'],
                         default='background_flagged_by_association')
TYPE_ORDER = ['actual_duplicate_source_record', 'cross_split_contamination_candidate', 'label_conflict_cluster', 'positive_repeat', 'event_window_repeat_same_market', 'event_window_repeat_cross_market', 'background_flagged_by_association']
TYPE_TAB = Bd.groupby(['b_type', 'split']).size().unstack(fill_value=0).reindex(columns=list(SPL), fill_value=0).reindex(TYPE_ORDER).fillna(0).astype(int); TYPE_TAB['total'] = TYPE_TAB.sum(axis=1)
TYPE_TAB['positives'] = Bd.groupby('b_type')['gt'].sum().reindex(TYPE_ORDER).fillna(0).astype(int)
TYPE_TAB['inactive_rows'] = Bd.groupby('b_type')['inactive'].sum().reindex(TYPE_ORDER).fillna(0).astype(int)
TYPE_TAB['distinct_vectors'] = Bd.groupby('b_type')['h'].nunique().reindex(TYPE_ORDER).fillna(0).astype(int)
assert int(TYPE_TAB['total'].sum()) == N_B
display(TYPE_TAB)
BASSOC = Bd[Bd.b_type == 'background_flagged_by_association']
ASSOC_DETAIL = {'rows': int(len(BASSOC)), 'inactive_rows': int(BASSOC.inactive.sum()), 'rows_with_imputed_feature': int((BASSOC.n_missing >= 1).sum()),
                'rows_in_clusters_with_more_than_K_copies': int((BASSOC.cluster_n > CFG['dedup_cap_K']).sum()), 'distinct_vectors': int(BASSOC.h.nunique()), 'positives': int(BASSOC['gt'].sum())}
print('background_flagged_by_association detail:', ASSOC_DETAIL)

# top clusters
CL = Bd.groupby('h').agg(rows=('h', 'size'), splits=('cluster_nsplits', 'first'), coins=('cluster_ncoins', 'first'), positives=('gt', 'sum'), event_rows=('is_event', 'sum'),
                          inactive_share=('inactive', 'mean'), dup_source_rows=('f_identical_source_record', 'sum'), first=('date', 'min'), last=('date', 'max'),
                          main_type=('b_type', lambda v: v.value_counts().index[0])).sort_values('rows', ascending=False)
print('Top-15 B vector clusters:'); display(CL.head(15))
CONC = {'b_distinct_vectors': int(len(CL)), 'b_rows_in_top10_clusters': int(CL.rows.head(10).sum()), 'share_in_top10': float(CL.rows.head(10).sum() / N_B), 'share_in_clusters_gt_K': float(CL.rows[CL.rows > CFG['dedup_cap_K']].sum() / N_B)}
print(CONC)

### 2c · Cross-split contamination (what actually matters for evaluation)
For each evaluation split: how many rows have a vector identical to some *train* vector, split by B type, and whether the matching train rows carry the same label. These are reported, not used to remove anything.

In [ ]:
trh = G.loc[G.split == 'train'].groupby('h')['gt'].agg(tr_n='size', tr_pos='sum')
CONTAM = {}
for sp in ('val', 'test'):
    e = G[G.split == sp].join(trh, on='h'); m = e.tr_n.notna()
    e['b_type'] = Bd.b_type.reindex(e.index)
    same = m & (((e['gt'] == 1) & (e.tr_pos > 0)) | ((e['gt'] == 0) & (e.tr_pos < e.tr_n)))
    CONTAM[sp] = {'rows': len(e), 'rows_vector_in_train': int(m.sum()), 'share_vector_in_train': float(m.mean()), 'positives': int(e['gt'].sum()), 'positives_vector_in_train': int((m & (e['gt'] == 1)).sum()),
                  'rows_with_label_conflict_vs_train': int((m & ~same).sum()), 'rows_identical_source_record_in_train': int((e.srec_h.isin(set(G.loc[G.split == 'train', 'srec_h']))).sum()),
                  'b_rows': int(e.b_type.notna().sum()), 'b_rows_vector_in_train': int((m & e.b_type.notna()).sum()),
                  'b_rows_vector_in_train_by_type': e[m & e.b_type.notna()].b_type.value_counts().to_dict()}
print(json.dumps(CONTAM, indent=1, default=jsonable))

### 2d · Optional TRAIN-ONLY filtering (pre-registered rule; no validation/test labels, no vector-identity deletion of legitimate rows)
| Rule | Rows dropped from **train** | Uses |
|---|---|---|
| **S1** | later copies of an identical source record inside train | raw record hash |
| **S2** | train copies of a source record that also occurs in val/test (decontamination) | raw record hash, split membership only |
| **S3** | for a train vector with more than K=20 train copies, **all** of whose train copies are inactive background (all activity features 0/missing), none positive and none event rows: keep K evenly spaced copies, drop the rest | train rows and train labels only |

Event rows, positive rows, label-mixed clusters and anything active are **never** touched by S3. Exemplars remain for every vector, so exact-vector overlap with validation/test persists by construction (it is reported below, not hidden). S3 is created only if it removes ≥ 1% of train rows.

In [ ]:
K = CFG['dedup_cap_K']
tr = G[G.split == 'train'].copy()
tr['tr_rank'] = tr.groupby('h').cumcount(); tr['tr_n'] = tr.groupby('h')['h'].transform('size')
tr['tr_pos'] = tr.groupby('h')['gt'].transform('sum'); tr['tr_event'] = tr.groupby('h')['is_event'].transform('max'); tr['tr_all_inactive'] = tr.groupby('h')['inactive'].transform('min')
S1 = (tr.srec_n >= 2) & (tr.srec_rank > 0)
S2 = (tr.srec_n >= 2) & (tr.srec_rank == 0) & tr.srec_in_eval
cand = (tr.tr_n > K) & tr.tr_all_inactive.astype(bool) & (tr.tr_pos == 0) & ~tr.tr_event.astype(bool)
r, n = tr.tr_rank.to_numpy(np.int64), tr.tr_n.to_numpy(np.int64)
keep_cap = (r == 0) | (((r * K) // n) != (((r - 1) * K) // n))
S3 = cand & ~pd.Series(keep_cap, index=tr.index)
rec = S1 | S2; rec_cap = S1 | S2 | S3
n_tr = len(tr)
def opt_counts(drop):
    k = tr[~drop]; y = k['gt'].to_numpy()
    return {'train_rows': int(len(k)), 'positives': int(y.sum()), 'negatives': int((1 - y).sum()), 'rows_dropped': int(drop.sum()), 'share_dropped': float(drop.mean()), 'positives_dropped': int(tr.loc[drop, 'gt'].sum()),
            'scale_pos_weight': float((len(k) - y.sum()) / y.sum()) if y.sum() else None}
OPTS = {'none_primary': opt_counts(pd.Series(False, index=tr.index)), 'records_only (S1+S2)': opt_counts(rec), 'records_plus_cap (S1+S2+S3)': opt_counts(rec_cap)}
S3_SHARE = float(S3.mean())
print('S1 rows:', int(S1.sum()), '| S2 rows:', int(S2.sum()), '| S3 candidates (rows in cappable clusters):', int(cand.sum()), '| S3 drops:', int(S3.sum()), f'({S3_SHARE:.2%} of train)')
display(pd.DataFrame(OPTS).T)

keys = {opt: set(tr.loc[~d, 'h']) for opt, d in {'none_primary': pd.Series(False, index=tr.index), 'records_only (S1+S2)': rec, 'records_plus_cap (S1+S2+S3)': rec_cap}.items()}
RESID = {}
for opt, ks in keys.items():
    RESID[opt] = {sp: {'rows_vector_in_filtered_train': int(G.loc[G.split == sp, 'h'].isin(ks).sum()), 'share': float(G.loc[G.split == sp, 'h'].isin(ks).mean()),
                       'positives_vector_in_filtered_train': int(G.loc[(G.split == sp) & (G['gt'] == 1), 'h'].isin(ks).sum())} for sp in ('val', 'test')}
print('residual val/test exact-vector overlap with the (filtered) train set (unchanged by design: exemplars remain):'); print(json.dumps(RESID, indent=1))
CAP_CREATED = bool(S3_SHARE >= CFG['dedup_min_share_of_train'])
REC_CREATED = bool(rec.sum() > 0)
SAP_FILTERS = {}
if REC_CREATED: SAP_FILTERS['records_only'] = tr.loc[rec, 'row_id'].to_numpy(np.int32)
if CAP_CREATED: SAP_FILTERS['records_plus_cap'] = tr.loc[rec_cap, 'row_id'].to_numpy(np.int32)
print('filters created:', {k: int(len(v)) for k, v in SAP_FILTERS.items()} or 'NONE (pre-registered thresholds not met -> primary unfiltered benchmark is the only configuration)')
if not CAP_CREATED: print(f'S3 below the pre-registered {CFG["dedup_min_share_of_train"]:.0%} of train -> NOT created (documented only).')

In [ ]:
# per-row flags (sidecar-like, additive; never replaces the v1 sidecar)
SAP_FLAGS = {}
btype = Bd.b_type.reindex(G.index); breason = Bd.b_primary_reason.reindex(G.index)
G['b_type'] = btype.astype('string'); G['b_primary_reason'] = breason.astype('string')
G['drop_records'] = False; G['drop_records_plus_cap'] = False; G['s3_candidate'] = False
G.loc[tr.index, 'drop_records'] = rec.to_numpy(); G.loc[tr.index, 'drop_records_plus_cap'] = rec_cap.to_numpy(); G.loc[tr.index, 's3_candidate'] = cand.to_numpy()
for sp in SPL:
    x = G[G.split == sp]
    SAP_FLAGS[sp] = pd.DataFrame({'row_id': x.row_id.to_numpy(np.int32), 'cluster_n': x.cluster_n.to_numpy(np.int32), 'cluster_nsplits': x.cluster_nsplits.to_numpy(np.int8), 'srec_n': x.srec_n.to_numpy(np.int32),
                                  'srec_rank': x.srec_rank.to_numpy(np.int32), 'inactive': x.inactive.to_numpy(bool), 'dup_category': x.dup_category.astype('string').to_numpy(), 'b_primary_reason': x.b_primary_reason.to_numpy(),
                                  'b_type': x.b_type.to_numpy(), 's3_candidate': x.s3_candidate.to_numpy(bool), 'drop_records': x.drop_records.to_numpy(bool), 'drop_records_plus_cap': x.drop_records_plus_cap.to_numpy(bool)})
SAP_INVEST = {'category_table': CAT_TAB.reset_index().to_dict(orient='records'), 'b_rows': N_B, 'independent_flags': INDEP.reset_index().to_dict(orient='records'),
              'exclusive_attribution': EXCL_ATTR.reset_index().to_dict(orient='records'), 'only_by_association_with_event_rows': ONLY_ASSOC, 'b_types': TYPE_TAB.reset_index().to_dict(orient='records'),
              'background_flagged_by_association_detail': ASSOC_DETAIL, 'concentration': CONC, 'top_clusters': CL.head(15).reset_index(drop=True).assign(first=lambda d: d['first'].astype(str), last=lambda d: d['last'].astype(str)).to_dict(orient='records'),
              'cross_split_contamination': CONTAM, 'filter_options': OPTS, 'residual_overlap': RESID, 'cap_K': K, 'cap_created': CAP_CREATED, 'records_filter_created': REC_CREATED}
print('Sapienza investigation complete.')

## 3 · Write the NEW artifacts (staged) and addendum manifests
The v1 manifests are **referenced by path + SHA-256, never edited**. Frozen v3/v1 files are not copied.

In [ ]:
STAGE.mkdir(parents=True, exist_ok=False)
SD, GD = STAGE / 'sapienza', STAGE / 'mtgox'
for d in ('reports', 'model_data'): (SD / d).mkdir(parents=True); (GD / d).mkdir(parents=True)
(GD / 'preprocessing').mkdir(parents=True)
REF_M1 = {ds: {'path': str(V1 / ds / 'training_manifest.json'), 'sha256': PIN_V1[ds + '/training_manifest.json']['sha256'], 'verdict_in_v1': M1[ds]['verdict']} for ds in M1}

# ---------------- Sapienza ----------------
for sp in SPL: SAP_FLAGS[sp].to_parquet(SD / 'model_data' / f'duplicate_row_flags_{sp}.parquet', index=False)
SAP_FILTER_FILES = {}
for k, ids in SAP_FILTERS.items():
    p = SD / 'model_data' / f'train_filter_{k}.parquet'; pd.DataFrame({'row_id': ids}).to_parquet(p, index=False); SAP_FILTER_FILES[k] = p.name
dump_json(SAP_INVEST, SD / 'reports' / 'b_duplicate_investigation.json')
SAP_REC = {'primary_benchmark': 'UNFILTERED train/val/test exactly as frozen v3 (class weights from the full train)',
           'sensitivity_runs_allowed': [k for k in SAP_FILTERS], 'selection_rule': 'choose among {primary, filters} by VALIDATION PR-AUC only; report test once, for the chosen configuration and for the primary',
           'caveat': 'exact-vector overlap of val/test with train persists under every filter (exemplars are kept); filters change only the training weight of repeated inactive/identical-record rows'}
SAP_ADD = {'artifact': 'sapienza targeted-improvements addendum', 'refers_to_v1_manifest': REF_M1['sapienza'], 'verdict_unchanged': M1['sapienza']['verdict'], 'frozen_v3_inputs_pin': {k: v['sha256'] for k, v in PIN_V3.items() if k.startswith('sapienza')},
           'duplicate_row_flags': {sp: f'model_data/duplicate_row_flags_{sp}.parquet' for sp in SPL}, 'optional_train_filters': {k: {'file': SAP_FILTER_FILES[k], 'rule': 'S1+S2' if k == 'records_only' else 'S1+S2+S3 (cap K=%d)' % K, **OPTS['records_only (S1+S2)' if k == 'records_only' else 'records_plus_cap (S1+S2+S3)']} for k in SAP_FILTERS},
           'filters_not_created': ([] if CAP_CREATED else ['records_plus_cap: S3 removes %.2f%% of train (< %.0f%% pre-registered threshold)' % (100 * S3_SHARE, 100 * CFG['dedup_min_share_of_train'])]) + ([] if REC_CREATED else ['records_only: no identical source records found']),
           'row_counts': OPTS, 'validation_and_test_filtered': False, 'recommendation': SAP_REC}
dump_json(SAP_ADD, SD / 'training_manifest_addendum.json')

# ---------------- Mt.Gox ----------------
dump_json({'columns': COL, 'within_conflict_cluster': WIN, 'single_column_conflict_stats': SINGLE, 'gates': GATES, 'admitted': ADMITTED, 'conflict_table': CONFLICT_TABLE, 'decision_rows': f'rows 0..{DECISION_MAX_ROW} (< validation end {B_IDX}); test labels unused',
           'cfg': CFG}, GD / 'reports' / 'extra_columns_assessment.json')
dump_json({'provenance': PROVENANCE, 'verdict': GOX_VERDICT, 'verdict_reasons': GOX_VERDICT_REASONS, 'v1_classification': PRIOR.get('classification')}, GD / 'reports' / 'provenance_and_verdict.json')
GOX_FILES = {}
if EXP is not None:
    for sp in SPL: pd.DataFrame(EXP['arrays'][sp], columns=['x_' + c for c in EXP['columns']]).to_parquet(GD / 'model_data' / f'experimental_extra_{sp}.parquet', index=False)
    joblib.dump({'columns': EXP['columns'], 'imputer': EXP['imputer'], 'scaler': EXP['scaler'], 'fitted_on': f'source rows 0..{A_IDX-1} (train only)'}, GD / 'preprocessing' / 'experimental_extra_preprocessing.joblib')
    GOX_FILES = {sp: f'model_data/experimental_extra_{sp}.parquet' for sp in SPL}
GOX_CONFIGS = {'A_baseline_7': {'columns': list(GOX_FEATS), 'source': 'frozen v3 parquets', 'status': 'primary, unchanged'}}
if EXP is not None:
    GOX_CONFIGS['X_baseline_plus_admitted'] = {'columns': list(GOX_FEATS) + ['x_' + c for c in EXP['columns']], 'n': len(GOX_FEATS) + len(EXP['columns']),
                                               'source': 'frozen v3 parquet columns + experimental_extra_{split}.parquet (row-aligned by position; same row counts)', 'status': 'EXPERIMENTAL ONLY: availability at prediction time unverified'}
dump_json(GOX_CONFIGS, GD / 'model_data' / 'feature_configs_experimental.json')
GOX_REC = {'primary': 'A_baseline_7', 'experimental': (['X_baseline_plus_admitted'] if EXP is not None else []), 'verdict': GOX_VERDICT,
           'reporting_rule': 'report as exploratory/source-specific; the experimental config only as a sensitivity run; never as evidence of deployable predictive skill',
           'conflict_note': 'conflicting-label clusters are not relabelled or majority-voted; evaluate on rows with a non-conflicting vector as a secondary view (see v1 row flags)'}
GOX_ADD = {'artifact': 'mtgox targeted-improvements addendum', 'refers_to_v1_manifest': REF_M1['mtgox'], 'verdict': GOX_VERDICT, 'verdict_reasons': GOX_VERDICT_REASONS, 'admitted_columns': ADMITTED, 'rejected_columns': {c: GATES[c]['rejection_reasons'] for c in PRESENT if c not in ADMITTED}, 'absent_columns': ABSENT,
           'configs': GOX_CONFIGS, 'experimental_files': GOX_FILES, 'conflicts': CONFLICT_TABLE, 'recommendation': GOX_REC}
dump_json(GOX_ADD, GD / 'training_manifest_addendum.json')

README = '\n'.join(['# processed_layer2_targeted_improvements_v1', '', 'Additive, versioned. Nothing here modifies processed_layer2_final_v3 or processed_layer2_model_ready_v1.', '',
                    '- sapienza/reports/b_duplicate_investigation.json: why the category-B rows were flagged, re-typing, contamination, filter options', '- sapienza/model_data/: per-row duplicate flags (+ optional training-only filters, if created)',
                    '- mtgox/reports/: assessment of buyer_7d_trade_count, buyer_7d_btc_sum, buyer_all_btc_sum; provenance and verdict', '- mtgox/model_data/: experimental extra-column parquets (if any column was admitted); feature_configs_experimental.json',
                    '- */training_manifest_addendum.json: references the v1 manifests (path + SHA-256), never edits them', '', 'Mt.Gox verdict: ' + GOX_VERDICT, 'No model was trained.'])
(STAGE / 'README.md').write_text(README, encoding='utf-8')
print('staged:', sorted(str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file()))

## 4 · TASK 3 — Validation of the NEW artifacts only
Every check is a hard assertion except those explicitly marked informational. Failure aborts before publication (the staging directory is left for inspection).

In [ ]:
CHK = []
def chk(name, ok, detail=''):
    CHK.append({'check': name, 'pass': bool(ok), 'detail': str(detail)[:200]}); assert ok, f'VALIDATION FAILED: {name} {detail}'

# 1 frozen inputs unchanged
chk('frozen v3 unchanged (sha256/size/mtime)', pin_dir(V3) == PIN_V3); chk('model-ready v1 unchanged', pin_dir(V1) == PIN_V1)
chk('raw sources unchanged (size, mtime)', {p.name: (p.stat().st_size, p.stat().st_mtime_ns) for p in (SAP_FILE, GOX_FILE)} == PIN_SRC)
chk('output folder still does not exist (nothing overwritten)', not OUT_FINAL.exists())

# 2 Sapienza flags
for sp in SPL:
    f = pd.read_parquet(SD / 'model_data' / f'duplicate_row_flags_{sp}.parquet')
    chk(f'flags rows == frozen rows ({sp})', len(f) == len(SAP_PQ[sp]), len(f)); chk(f'flags row_id positional ({sp})', np.array_equal(f.row_id.to_numpy(), np.arange(len(f))))
    chk(f'flags dup_category == v1 sidecar ({sp})', (f.dup_category.astype(str).to_numpy() == SC[sp]['dup_category'].astype(str).to_numpy()).all())
    chk(f'b_type present iff category B ({sp})', ((f.b_type.notna()) == (f.dup_category == 'B_potentially_harmful')).all())
chk('B attribution sums to B total (exclusive reasons and types)', int(EXCL_ATTR.total.sum()) == N_B == int(TYPE_TAB.total.sum()), N_B)
chk('B count equals v1 sidecar B count', N_B == int(sum((SC[sp]['dup_category'] == 'B_potentially_harmful').sum() for sp in SPL)))

# 3 filter safety + independent recomputation (sidecar columns only; fresh code path)
T = SC['train']; Tn = len(T)
hh = T['vec_hash'].to_numpy(np.uint64); yy = T['gt'].to_numpy(np.int8); ev = (T['pump_index'].to_numpy(float) >= 0)
d = pd.DataFrame({'h': hh, 'y': yy, 'ev': ev}); d['n'] = d.groupby('h')['h'].transform('size'); d['pos'] = d.groupby('h')['y'].transform('sum'); d['evc'] = d.groupby('h')['ev'].transform('max'); d['r'] = d.groupby('h').cumcount()
fl = pd.read_parquet(SD / 'model_data' / 'duplicate_row_flags_train.parquet')
inact_cluster = pd.Series(fl.inactive.to_numpy()).groupby(hh).transform('min').to_numpy().astype(bool)
cand2 = (d.n > K).to_numpy() & inact_cluster & (d.pos == 0).to_numpy() & ~d.evc.to_numpy().astype(bool)
keep2 = (d.r == 0).to_numpy() | (((d.r * K) // d.n) != (((d.r - 1) * K) // d.n)).to_numpy()
s3_2 = cand2 & ~keep2
chk('S3 recomputed independently from sidecar == saved flag', np.array_equal(s3_2, (fl.drop_records_plus_cap.to_numpy() & ~fl.drop_records.to_numpy())) or np.array_equal(s3_2 & ~fl.drop_records.to_numpy(), fl.drop_records_plus_cap.to_numpy() & ~fl.drop_records.to_numpy()))
chk('S3 never drops a positive or an event row', not (s3_2 & ((yy == 1) | ev)).any())
chk('S3 candidates carry no positive in any train copy', (d.pos[cand2] == 0).all())
for k, fn in SAP_FILTER_FILES.items():
    ids = pd.read_parquet(SD / 'model_data' / fn).row_id.to_numpy()
    chk(f'filter {k}: ids unique and inside the train range', len(np.unique(ids)) == len(ids) and (len(ids) == 0 or (ids.min() >= 0 and ids.max() < Tn)))
    flag_col = 'drop_records' if k == 'records_only' else 'drop_records_plus_cap'
    chk(f'filter {k}: file == flag column', np.array_equal(np.sort(ids), np.flatnonzero(fl[flag_col].to_numpy())))
    kept = np.ones(Tn, bool); kept[ids] = False
    chk(f'filter {k}: class counts recomputed', (int(kept.sum()), int(yy[kept].sum())) == (OPTS['records_only (S1+S2)' if k == 'records_only' else 'records_plus_cap (S1+S2+S3)']['train_rows'], OPTS['records_only (S1+S2)' if k == 'records_only' else 'records_plus_cap (S1+S2+S3)']['positives']))
chk('val/test flags carry no drop', not any(pd.read_parquet(SD / 'model_data' / f'duplicate_row_flags_{sp}.parquet')[['drop_records', 'drop_records_plus_cap']].to_numpy().any() for sp in ('val', 'test')))
chk('filters were not chosen with val/test labels (rule inputs: train hashes, train labels, raw record hash, split membership only)', True, 'by construction; see 2d')
chk('primary benchmark rows unchanged', OPTS['none_primary']['train_rows'] == len(SAP_PQ['train']) == Tn)

# 4 Mt.Gox experimental artifacts
if EXP is not None:
    cols = EXP['columns']; raw = pd.read_csv(GOX_FILE, usecols=cols, dtype='float64')
    for sp, (lo, hi) in BOUNDS.items():
        ex = pd.read_parquet(GD / 'model_data' / f'experimental_extra_{sp}.parquet')
        chk(f'experimental rows == v3 rows ({sp})', len(ex) == hi - lo, len(ex))
        chk(f'experimental values recomputed from raw ({sp})', np.array_equal(ex.to_numpy(np.float32), EXP['scaler'].transform(EXP['imputer'].transform(raw.iloc[lo:hi][cols].to_numpy())).astype(np.float32)))
    tr_raw = raw.iloc[:A_IDX][cols].to_numpy(); med = np.nanmedian(tr_raw, axis=0); filled = np.where(np.isnan(tr_raw), med, tr_raw)
    q1, q3 = np.percentile(filled, [25, 75], axis=0); sc_n = q3 - q1; sc_n[sc_n == 0] = 1
    chk('experimental imputer median == train-only numpy recompute', np.allclose(EXP['imputer'].statistics_, med, rtol=1e-9, atol=0))
    chk('experimental scaler == train-only numpy recompute', np.allclose(EXP['scaler'].center_, np.median(filled, axis=0), rtol=1e-9) and np.allclose(EXP['scaler'].scale_, sc_n, rtol=1e-9))
    jb = joblib.load(GD / 'preprocessing' / 'experimental_extra_preprocessing.joblib'); chk('saved scaler loads and equals in-memory', np.array_equal(jb['scaler'].scale_, EXP['scaler'].scale_)); del raw
else:
    chk('no experimental files written when nothing admitted', not list((GD / 'model_data').glob('experimental_extra_*')))
chk('baseline parquets/joblibs not duplicated into the new folder', not any(p.name.startswith(('mtgox_', 'sapienza_')) and p.suffix in ('.parquet', '.joblib') for p in STAGE.rglob('*') if p.is_file()))

# 5 gate reproducibility and test-label isolation
for c in PRESENT:
    x = GX[c].to_numpy(float)
    chk(f'gate AUC reproducible: {c}', np.isclose(auc_rank(x[TR_IDX], Y[TR_IDX]), COL[c]['leakage']['auc_train_sample'], equal_nan=True) and np.isclose(auc_rank(x[VA_IDX], Y[VA_IDX]), COL[c]['leakage']['auc_val_sample'], equal_nan=True))
chk('decision rows all before the test partition', DECISION_MAX_ROW < B_IDX, DECISION_MAX_ROW)
chk('admitted columns passed every leakage and reliability gate', all(GATES[c]['safe'] for c in ADMITTED))
chk('verdict remains EXPLORATORY unless chronology and label definition both documented', GOX_VERDICT.startswith('EXPLORATORY'))
chk('conflict table: admitted never worse than baseline fragmentation bookkeeping', all(v.get('still_conflicting_rows', 0) <= BASE_CONFLICT['decision_rows(train+val)']['conflicting_rows'] for k, v in CONFLICT_TABLE.items() if 'train+val' in k and isinstance(v, dict) and 'still_conflicting_rows' in v))

# 6 readability
for p in STAGE.rglob('*.json'): json.loads(p.read_text())
for p in STAGE.rglob('*.parquet'): pq.read_metadata(p)
chk('all staged JSON/parquet readable', True)
print(f'{sum(c["pass"] for c in CHK)}/{len(CHK)} validations passed')

## 5 · Publish (atomic) and SHARE BLOCK

In [ ]:
VAL = {'passed': sum(c['pass'] for c in CHK), 'total': len(CHK), 'checks': CHK}
dump_json({'run_started_utc': RUN_STARTED, 'validation': VAL, 'cfg': CFG, 'verdicts': {'mtgox': GOX_VERDICT, 'sapienza': M1['sapienza']['verdict']}, 'mock_or_real': 'real data if executed on the Drive'}, STAGE / 'VALIDATION_REPORT.json')
assert pin_dir(V3) == PIN_V3 and pin_dir(V1) == PIN_V1, 'frozen inputs changed during the run'
assert not OUT_FINAL.exists(); os.rename(STAGE, OUT_FINAL)
print('PUBLISHED ->', OUT_FINAL)

bt = TYPE_TAB['total'].to_dict(); bc = BASE_CONFLICT['decision_rows(train+val)']; ad = CONFLICT_TABLE['plus_admitted_columns(train+val rows)'] if ADMITTED else None
SHARE = {'task3_validation': f"{VAL['passed']}/{VAL['total']}",
         'mtgox': {'verdict': GOX_VERDICT, 'columns_present': PRESENT, 'columns_absent': ABSENT, 'admitted_experimental': ADMITTED,
                   'rejected': {c: GATES[c]['rejection_reasons'] for c in PRESENT if c not in ADMITTED},
                   'availability': {c: COL[c]['availability_at_prediction_time'][:40] for c in PRESENT},
                   'auc_train_val': {c: [round(COL[c]['leakage']['auc_train_sample'], 3), round(COL[c]['leakage']['auc_val_sample'], 3)] for c in PRESENT},
                   'baseline_conflicting_rows_train+val': bc['conflicting_rows'], 'baseline_conflicting_clusters_all_rows': BASE_CONFLICT['all_rows(descriptive)']['conflicting_clusters'],
                   'remaining_conflicting_rows_with_admitted': (ad or {}).get('still_conflicting_rows'), 'excess_resolved_vs_null': (ad or {}).get('excess_share_resolved_vs_null'),
                   'chronology': CHRONO_SUPPORTED, 'label_doc': LABEL_DOC_FOUND, 'participant_ids': bool(PARTICIPANT)},
         'sapienza': {'B_rows': N_B, 'only_by_association_with_event_rows': ONLY_ASSOC, 'B_types': bt, 'actual_duplicate_source_records_in_B': bt['actual_duplicate_source_record'],
                      'cross_split_contamination_candidates': bt['cross_split_contamination_candidate'], 'positives_in_B': int(Bd['gt'].sum()),
                      'filters': {k: OPTS[{'records_only': 'records_only (S1+S2)', 'records_plus_cap': 'records_plus_cap (S1+S2+S3)'}[k]] for k in SAP_FILTERS}, 'primary': OPTS['none_primary'], 'cap_created': CAP_CREATED}}
print('\n==================== SHARE BLOCK ====================')
print(json.dumps(SHARE, indent=1, default=jsonable))
print('\nRECOMMENDATIONS FOR XGBOOST (no model trained here):')
print(' Sapienza : primary = unfiltered config A with full-train class weight; filters only as validation-selected sensitivity runs; test reported once.')
print(' Mt.Gox   : primary = 7-feature baseline; experimental config (if any) only as a sensitivity run; report as EXPLORATORY, never as deployable skill.')
print('=====================================================')